# 04 — Test Deep Agents Framework

Goal: Verify the `deepagents` package works end-to-end:
- Create a coordinator with a subagent
- Subagent uses a tool
- Coordinator delegates work and returns results

This uses `ollama:llama3` (or change to your available model below).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
os.environ["AGENT_MODEL"] = os.getenv("AGENT_MODEL", "ollama:llama3")

## 4.1 Create a simple tool

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class AnalysisResult(BaseModel):
    scenes: list[dict] = Field(default_factory=list)
    characters: list[dict] = Field(default_factory=list)
    style_notes: str = ""


def analyze_anime_clip(clip_path: str) -> dict:
    """Analyze an anime clip and return scene/character data."""
    return {
        "scenes": [
            {"id": 1, "start": 0.0, "end": 5.5, "description": "Hero arrives at the ruined temple"}
        ],
        "characters": [
            {"name": "Hero", "appearance": "Spiky dark hair, red eyes, black cloak"}
        ],
        "style_notes": "Dark fantasy aesthetic with high contrast lighting.",
    }


print("Tool and schema defined OK")

## 4.2 Create a subagent

In [ ]:
from deepagents import create_deep_agent

model = os.environ["AGENT_MODEL"]
print(f"Using model: {model}")

agent = create_deep_agent(
    model=model,
    system_prompt="You are a coordinator. Use the analyze-agent subagent to process requests.",
    subagents=[
        {
            "name": "analyze-agent",
            "description": "Analyzes anime clips and returns structured data.",
            "system_prompt": "You are an anime analysis specialist.",
            "tools": [analyze_anime_clip],
            "response_format": AnalysisResult,
        }
    ],
    name="test-coordinator",
)
print(f"Agent created: {agent.name if hasattr(agent, 'name') else type(agent).__name__}")

## 4.3 Run the agent

In [ ]:
try:
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": "Analyze the anime clip at /path/to/clip.mp4"}
            ]
        },
        config={"configurable": {"thread_id": "test-01"}},
    )
    print("Invocation succeeded!")
    print("Result messages:")
    for m in result.get("messages", []):
        print(f"  [{m.type if hasattr(m, 'type') else type(m).__name__}]: {str(m.content)[:200]}")
except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()

## 4.4 Test the full pipeline creation (import from project)

In [ ]:
import sys
sys.path.insert(0, "..")

from app.agents.pipeline import create_pipeline_agent

try:
    pipeline = create_pipeline_agent()
    print("Pipeline agent created OK")
    print(f"Type: {type(pipeline).__name__}")
except Exception as e:
    print(f"ERROR creating pipeline agent: {e}")
    import traceback
    traceback.print_exc()

## Summary

| Check | Status |
|-------|--------|
| Tool creation works | ⬜ |
| Subagent creation works | ⬜ |
| Coordinator invokes subagent | ⬜ |
| Pipeline agent imports & creates | ⬜ |

If any step fails, the issue is likely the LLM backend (Ollama not running, wrong model name, etc.).